# RunPod Execution: Robust NN Experiments

---

## Pre-flight checklist
| # | Action |
|---|--------|
| 1 | Upload **this notebook** + `12April2026_RobustNN_Experiments.py` to `/workspace/` |
| 2 | Select a RunPod template with **PyTorch 2.x** (CUDA 12.1+), e.g. `runpod/pytorch:2.4.0-py3.11-cuda12.4.1-devel` |
| 3 | Click **Kernel → Restart Kernel and Run All Cells** |
| 4 | **Before stopping the pod**: run the last cell to zip and download results |

---

## What gets produced
All outputs are saved to `/workspace/results_12April2026/`:
- **CSV files**: noise results, FGSM results, SDIV surface, NLP results, CLIP results  
- **PNG plots**: training curves (per-loss subplots), normalized confusion matrices,  
  robustness curves, 3D SDIV surface, curriculum annealing, dual frontier scatter  
- **Final ZIP**: `runpod_results_12April2026.zip` (download before stopping pod!)

---

> **Security**: Do NOT paste HuggingFace tokens here. Use RunPod environment variables.

## Cell 1 — Install all dependencies (pinned, RunPod-safe)

**What this does:**
- Installs all required packages with version pins tuned for RunPod CUDA 12.x images
- `--root-user-action=ignore` suppresses the RunPod pip root-user warning
- Re-installing is safe — pip checks existing versions and skips if already satisfied
- `typing_extensions>=4.10` must be pinned FIRST to avoid `torch` import errors on some images

**Known RunPod issues this fixes:**
- `ImportError: cannot import name 'override' from 'typing_extensions'` → fixed by upgrading `typing_extensions` before torch
- `ModuleNotFoundError: No module named 'medmnist'` → installed here
- `open_clip_torch` not bundled in standard PyTorch images → installed here
- `kaleido` needed for plotly static image export → installed here

**After this cell completes:** you may see a message asking to restart the kernel.  
If you do, click **Restart** and then **Run All** again — the install is idempotent.

In [1]:
import sys, subprocess, importlib

def run_pip(*pkgs):
    """Install packages with pinned versions, suppressing RunPod root-user warning."""
    cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--root-user-action=ignore",
        *pkgs
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("INSTALL ERROR:")
        print(result.stderr[-3000:])
    return result.returncode == 0

# Step 1: typing_extensions MUST come first — avoids downstream torch import failure
print("[Step 1/5] Upgrading typing_extensions ...")
run_pip("typing_extensions>=4.10.0")

# Step 2: Core PyTorch stack — install with CUDA wheel (not plain PyPI CPU wheel)
print("[Step 2/5] Checking torch version and CUDA availability ...")
import torch, subprocess as _sp, re as _re

def _detect_cuda_tag():
    """Detect CUDA version from nvcc/nvidia-smi, return PyTorch wheel tag."""
    try:
        nvcc = _sp.check_output(["nvcc", "--version"], text=True, stderr=_sp.DEVNULL)
        m = _re.search(r"release (\d+)\.(\d+)", nvcc)
        if m:
            maj, mn = int(m.group(1)), int(m.group(2))
            if maj == 12 and mn >= 4: return "cu124"
            if maj == 12 and mn >= 1: return "cu121"
            if maj == 11 and mn >= 8: return "cu118"
    except Exception:
        pass
    try:
        smi = _sp.check_output(["nvidia-smi"], text=True, stderr=_sp.DEVNULL)
        m = _re.search(r"CUDA Version: (\d+)\.(\d+)", smi)
        if m:
            maj, mn = int(m.group(1)), int(m.group(2))
            if maj == 12 and mn >= 4: return "cu124"
            if maj == 12 and mn >= 1: return "cu121"
            if maj == 11 and mn >= 8: return "cu118"
    except Exception:
        pass
    return "cu121"  # safe default for modern RunPod H100/A100/A40 pods

tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
_has_cuda = torch.cuda.is_available()
_needs_reinstall = (tv < (2, 6)) or (not _has_cuda)

if not _has_cuda:
    print(f"  WARNING: torch {torch.__version__} has NO CUDA support (CPU-only build from PyPI)")
    print("  Root cause: plain pip install torch always gives CPU wheel.")
    print("  Fix: reinstalling with GPU/CUDA index URL ...")
elif tv < (2, 6):
    print(f"  torch {torch.__version__} is < 2.6 — UPGRADING with CUDA wheel ...")
else:
    print(f"  torch {torch.__version__} OK (>= 2.6, CUDA available — no reinstall needed)")

if _needs_reinstall:
    _cuda_tag = _detect_cuda_tag()
    _index_url = f"https://download.pytorch.org/whl/{_cuda_tag}"
    print(f"  Detected CUDA tag : {_cuda_tag}")
    print(f"  Wheel index URL   : {_index_url}")
    print("  Installing (3-5 min) ...")
    _cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--root-user-action=ignore",
        "--index-url", _index_url,
        "torch>=2.6.0", "torchvision>=0.21.0",
    ]
    _r = _sp.run(_cmd, capture_output=True, text=True)
    if _r.returncode != 0:
        print(f"    INSTALL ERROR: {_r.stderr[-2000:]}")
    else:
        print("  torch + CUDA wheel installed.")
    print("  IMPORTANT: torch was reinstalled — restart kernel now:")
    print("  Kernel -> Restart Kernel, then Run All Cells again.")
# Step 3: HuggingFace ecosystem
print("[Step 3/5] Installing HuggingFace + NLP packages ...")
ok = run_pip(
    "transformers>=4.40.0",
    "datasets>=2.18.0",
    "accelerate>=0.30.0",
    "huggingface_hub>=0.22.0",
)
print("  HuggingFace stack", "✓" if ok else "✗ (see error above)")

# Step 4: Vision / multimodal extras
print("[Step 4/5] Installing vision + multimodal packages ...")
ok = run_pip(
    "medmnist>=2.2.0",
    "open_clip_torch>=2.24.0",
    "Pillow>=10.0.0",
)
print("  Vision extras", "✓" if ok else "✗ (see error above)")

# Step 5: Science / plotting stack
print("[Step 5/5] Installing science + plotting packages ...")
ok = run_pip(
    "scikit-learn>=1.3.0",
    "matplotlib>=3.8.0",
    "seaborn>=0.13.0",
    "pandas>=2.1.0",
    "numpy>=1.24.0",
    "tqdm>=4.66.0",
    "plotly>=5.20.0",
    "kaleido>=0.2.1",
)
print("  Science/plotting stack", "✓" if ok else "✗ (see error above)")
print("\n=== Installation complete ===")
print("Python:", sys.version)
print("If packages were freshly installed, click Kernel → Restart and then Run All.")

[Step 1/5] Upgrading typing_extensions ...
[Step 2/5] Checking torch version ...
  torch 2.4.1+cu124 ✓ (no reinstall needed)
[Step 3/5] Installing HuggingFace + NLP packages ...
  HuggingFace stack ✓
[Step 4/5] Installing vision + multimodal packages ...
  Vision extras ✓
[Step 5/5] Installing science + plotting packages ...
  Science/plotting stack ✓

=== Installation complete ===
Python: 3.11.10 (main, Sep  7 2024, 18:35:41) [GCC 11.4.0]
If packages were freshly installed, click Kernel → Restart and then Run All.


## Cell 2 — GPU & version sanity check

**What this does:**
- Confirms the GPU is visible and reports VRAM
- Validates all key package versions are meet-or-exceed the minimums
- Detects if `torch.amp` (new API) or `torch.cuda.amp` (old API) should be used  
  → this is the most common silent compatibility bug on RunPod

**What to do if GPU shows as `None`:**
- Restart the RunPod pod and make sure you selected a GPU tier
- On RunPod, non-GPU instances have `torch.cuda.is_available() == False` → code falls back to CPU automatically

**Minimum requirements to run the full paper experiments:**
- 16 GB VRAM (L4/A10G) for Part A (Vision ViT full 250 epochs)
- 24 GB VRAM (A100/H100) recommended for Part B (BERT fine-tuning batch=32)

In [2]:
import importlib.metadata as imeta
import torch
import sys

# ── GPU report ─────────────────────────────────────────────────────────────────
print("="*60)
print("GPU STATUS")
print("="*60)
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    for i in range(n_gpus):
        d = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {d.name}")
        print(f"    VRAM  : {d.total_memory / 1024**3:.1f} GB")
        print(f"    SM    : {d.multi_processor_count} SMs")
    print(f"  CUDA version : {torch.version.cuda}")
    print(f"  PyTorch      : {torch.__version__}")
else:
    print("  ⚠ No GPU detected — running on CPU (very slow for ViT/BERT).")
    print("  Check RunPod pod settings.")

# ── AMP API detection ─────────────────────────────────────────────────────────
# torch>=2.4: torch.amp.GradScaler / torch.amp.autocast (preferred)
# torch<2.4:  torch.cuda.amp.GradScaler / torch.cuda.amp.autocast (deprecated but works)
tv = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
AMP_NEW_API = tv >= (2, 4)
print(f"\n  AMP API: {'torch.amp (new, >=2.4)' if AMP_NEW_API else 'torch.cuda.amp (legacy, <2.4)'}")
print(f"  AMP will be: {'ON (GPU available)' if torch.cuda.is_available() else 'OFF (CPU only)'}")

# ── Package version matrix ────────────────────────────────────────────────────
print("\n" + "="*60)
print("PACKAGE VERSIONS")
print("="*60)

MIN_VERSIONS = {
    "torch":              (2, 1, 0),
    "torchvision":        (0, 16, 0),
    "transformers":       (4, 40, 0),
    "datasets":           (2, 18, 0),
    "accelerate":         (0, 30, 0),
    "scikit-learn":       (1, 3, 0),
    "matplotlib":         (3, 8, 0),
    "seaborn":            (0, 13, 0),
    "pandas":             (2, 1, 0),
    "numpy":              (1, 24, 0),
    "tqdm":               (4, 66, 0),
    "medmnist":           (2, 2, 0),
    "open_clip_torch":    (2, 24, 0),
    "typing_extensions":  (4, 10, 0),
}

all_good = True
for pkg, (ma, mi, pa) in MIN_VERSIONS.items():
    try:
        ver_str = imeta.version(pkg)
        parts = [int(x) for x in ver_str.split('+')[0].split('.')[:3]]
        while len(parts) < 3:
            parts.append(0)
        ok = tuple(parts) >= (ma, mi, pa)
        mark = "✓" if ok else "✗ NEEDS UPDATE"
        if not ok:
            all_good = False
        print(f"  {pkg:<25} {ver_str:<15} {mark}")
    except imeta.PackageNotFoundError:
        print(f"  {pkg:<25} {'NOT INSTALLED':<15} ✗ MISSING")
        all_good = False

print("\n" + ("✓ All packages OK — ready to run experiments." if all_good
               else "✗ Some packages need updating — re-run Cell 1."))

GPU STATUS
  GPU 0: NVIDIA A40
    VRAM  : 44.4 GB
    SM    : 84 SMs
  CUDA version : 12.4
  PyTorch      : 2.4.1+cu124

  AMP API: torch.amp (new, >=2.4)
  AMP will be: ON (GPU available)

PACKAGE VERSIONS
  torch                     2.4.1+cu124     ✓
  torchvision               0.19.1+cu124    ✓
  transformers              5.5.3           ✓
  datasets                  4.8.4           ✓
  accelerate                1.13.0          ✓
  scikit-learn              1.8.0           ✓
  matplotlib                3.10.8          ✓
  seaborn                   0.13.2          ✓
  pandas                    3.0.2           ✓
  numpy                     2.4.4           ✓
  tqdm                      4.67.3          ✓
  medmnist                  3.0.2           ✓
  open_clip_torch           3.3.0           ✓
  typing_extensions         4.15.0          ✓

✓ All packages OK — ready to run experiments.


## Cell 3 — Workspace, paths, and environment setup

**What this does:**
- Sets `/workspace/` as the working directory (persistent RunPod storage)
- Configures `HF_HOME` so HuggingFace model weights cache inside `/workspace/`  
  (survives pod restarts if you have persistent storage enabled)
- Locates the companion `.py` script automatically
- Sets all `ROBUST_NN_*` environment variables that control the experiment

**About the environment variables:**
| Variable | Default | Meaning |
|----------|---------|----------|
| `ROBUST_NN_QUICK_RUN` | `0` | `1`=fast debug, `0`=full paper run |
| `ROBUST_NN_PART` | `ABC` | Which modalities: `A`=Vision, `B`=NLP, `C`=CLIP |
| `ROBUST_NN_VIT_EPOCHS` | `250` | ViT training epochs (paper: 250) |
| `ROBUST_NN_NLP_EPOCHS` | `15` | BERT fine-tune epochs (paper: 15) |
| `ROBUST_NN_RESULTS_DIR` | `results_12April2026` | Output folder name |

**Change `QUICK_RUN` to `0` and `VIT_EPOCHS` to `250` for the full paper run.**

In [3]:
import os
import sys
from pathlib import Path

# ── Working directory ──────────────────────────────────────────────────────────
WORKSPACE = Path("/workspace")
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE)
print(f"Working directory: {os.getcwd()}")

# ── Results directory (inside /workspace so it persists) ──────────────────────
RESULTS_DIR = WORKSPACE / "results_12April2026"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results directory: {RESULTS_DIR}")

# ── HuggingFace cache → /workspace (survives pod restart) ─────────────────────
HF_HOME = str(WORKSPACE / "hf_cache")
os.environ["HF_HOME"] = HF_HOME
os.environ["TRANSFORMERS_CACHE"] = HF_HOME
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / "hf_datasets_cache")
Path(HF_HOME).mkdir(parents=True, exist_ok=True)
Path(os.environ["HF_DATASETS_CACHE"]).mkdir(parents=True, exist_ok=True)
print(f"HF cache: {HF_HOME}")

# ── Experiment control environment variables ───────────────────────────────────
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  CHANGE THESE for full paper run:                                       │
# │    QUICK_RUN = "0"       (remove quick mode)                            │
# │    VIT_EPOCHS = "250"    (full training)                                │
# │    NLP_EPOCHS = "15"     (full fine-tuning)                             │
# └─────────────────────────────────────────────────────────────────────────┘
os.environ["ROBUST_NN_QUICK_RUN"]    = "0"       # ← set to "0" for full run
os.environ["ROBUST_NN_PART"]         = "ABC"     # A=Vision, B=NLP, C=CLIP
os.environ["ROBUST_NN_VIT_EPOCHS"]   = "10"      # ← set to "250" for paper
os.environ["ROBUST_NN_NLP_EPOCHS"]   = "5"       # ← set to "15" for paper
os.environ["ROBUST_NN_NLP_MAX_TRAIN"]= "1500"    # max NLP training samples
os.environ["ROBUST_NN_VIT_BATCH"]    = "64"
os.environ["ROBUST_NN_NLP_BATCH"]    = "32"
os.environ["ROBUST_NN_RESULTS_DIR"]  = str(RESULTS_DIR)
os.environ["ROBUST_NN_SEED"]         = "42"

# ── Locate companion .py script ───────────────────────────────────────────────
def find_script(fname: str) -> Path:
    """Search common RunPod upload locations for the .py script."""
    search = [
        Path.cwd(),
        Path("."),
        Path("/workspace"),
        Path("/workspace/code"),
        Path("/workspace/Robust-NN-learning/code"),
        Path("/workspace/Robust-NN-learning"),
    ]
    for base in search:
        p = base / fname
        if p.is_file():
            return p.resolve()
    raise FileNotFoundError(
        f"Cannot find '{fname}'.\n"
        f"Upload it to /workspace/ (same folder as this notebook).\n"
        f"Searched: {[str(b) for b in search]}"
    )

SCRIPT = find_script("12April2026_RobustNN_Experiments.py")
# Add script directory to path so relative imports work
if str(SCRIPT.parent) not in sys.path:
    sys.path.insert(0, str(SCRIPT.parent))

print(f"\nExperiment script: {SCRIPT}")
print("\nEnvironment config:")
for k, v in os.environ.items():
    if k.startswith("ROBUST_NN"):
        print(f"  {k} = {v}")
print("\n✓ Workspace setup complete. Ready to run experiments.")

Working directory: /workspace
Results directory: /workspace/results_12April2026
HF cache: /workspace/hf_cache

Experiment script: /workspace/12April2026_RobustNN_Experiments.py

Environment config:
  ROBUST_NN_QUICK_RUN = 0
  ROBUST_NN_PART = ABC
  ROBUST_NN_VIT_EPOCHS = 10
  ROBUST_NN_NLP_EPOCHS = 5
  ROBUST_NN_NLP_MAX_TRAIN = 1500
  ROBUST_NN_VIT_BATCH = 64
  ROBUST_NN_NLP_BATCH = 32
  ROBUST_NN_RESULTS_DIR = /workspace/results_12April2026
  ROBUST_NN_SEED = 42

✓ Workspace setup complete. Ready to run experiments.


## Cell 4 — AMP compatibility patch

**What this does:**
- Patches the `torch.cuda.amp` deprecation in PyTorch >= 2.4
- In PyTorch >= 2.4, `torch.cuda.amp.GradScaler` and `autocast` still work but trigger  
  `FutureWarning: torch.cuda.amp.GradScaler is deprecated`. This cell silences that.
- Also patches the experiment script's `autocast` call to use `device_type='cuda'`  
  argument format which is required in PyTorch 2.6+

**Why this matters:**
The `autocast(enabled=USE_AMP)` call in the .py file uses the old signature.  
PyTorch 2.6+ requires `torch.amp.autocast(device_type='cuda', enabled=True)`.  
This cell monkey-patches both to be compatible with all versions 2.1–2.6+.

**No code changes needed in `.py` file** — this patch runs once at import time.

In [4]:
import torch
import warnings

# Silence all FutureWarnings from torch.cuda.amp (deprecated in 2.4+)
warnings.filterwarnings("ignore", category=FutureWarning, module="torch")
warnings.filterwarnings("ignore", category=UserWarning, module="torch")

# ── AMP compatibility shim ────────────────────────────────────────────────────
# Ensures torch.cuda.amp.GradScaler and autocast work on all PyTorch 2.x versions.
# The .py script uses `from torch.cuda.amp import GradScaler, autocast`.
# On torch>=2.6 this still works but may warn. Patch it to use new API.

tv = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])

if tv >= (2, 4):
    # New API: torch.amp.GradScaler(device='cuda') and torch.amp.autocast(device_type='cuda')
    import torch.amp

    class _PatchedGradScaler(torch.amp.GradScaler):
        """GradScaler that works with both old (enabled=) and new (device=) API."""
        def __init__(self, enabled=True, **kwargs):
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
            super().__init__(device=device, enabled=enabled, **kwargs)

    import torch.cuda.amp as _cuda_amp
    _cuda_amp.GradScaler = _PatchedGradScaler

    # Patch autocast to add device_type automatically if not provided
    _orig_autocast = torch.cuda.amp.autocast
    class _PatchedAutocast:
        def __init__(self, enabled=True, dtype=None, **kwargs):
            device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
            kw = {k: v for k, v in kwargs.items() if k != 'device_type'}
            self._ctx = torch.amp.autocast(device_type=device_type, enabled=enabled, **kw)
        def __enter__(self): return self._ctx.__enter__()
        def __exit__(self, *a): return self._ctx.__exit__(*a)
    _cuda_amp.autocast = _PatchedAutocast

    print(f"✓ AMP patch applied (torch {torch.__version__} >= 2.4 — using new torch.amp API)")
else:
    print(f"✓ AMP: using legacy torch.cuda.amp API (torch {torch.__version__} < 2.4)")

# ── Verify AMP works ──────────────────────────────────────────────────────────
try:
    from torch.cuda.amp import GradScaler, autocast
    sc = GradScaler(enabled=False)  # enabled=False → no-op, safe on CPU
    with autocast(enabled=False):
        t = torch.ones(2, 2)
    print("✓ AMP imports verified (GradScaler + autocast OK)")
except Exception as e:
    print(f"✗ AMP verification failed: {e}")
    print("  The experiments will still run but may show warnings.")

✓ AMP patch applied (torch 2.4.1+cu124 >= 2.4 — using new torch.amp API)
✓ AMP imports verified (GradScaler + autocast OK)


## Cell 5 — Import all modules and verify

**What this does:**
- Imports every module needed before the experiment script runs
- Catches and reports import errors with actionable fix messages
- Checks for `medmnist`, `open_clip_torch`, `transformers`, `datasets` separately  
  (these are optional — if missing, only the relevant Part is skipped)
- Reports which Parts (A/B/C) are available to run

**If you see an ImportError:**
- Re-run Cell 1 to install missing packages
- Then restart kernel and re-run from Cell 1

In [5]:
import sys, importlib

IMPORT_STATUS = {}

def try_import(name, friendly_name=None):
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, '__version__', 'unknown')
        IMPORT_STATUS[name] = (True, ver)
        print(f"  ✓ {friendly_name or name:<30} {ver}")
        return True
    except ImportError as e:
        IMPORT_STATUS[name] = (False, str(e))
        print(f"  ✗ {friendly_name or name:<30} NOT FOUND — re-run Cell 1")
        return False

print("REQUIRED packages:")
req = all([
    try_import("torch"),
    try_import("torchvision"),
    try_import("numpy"),
    try_import("pandas"),
    try_import("matplotlib"),
    try_import("seaborn"),
    try_import("sklearn", "scikit-learn"),
    try_import("tqdm"),
    try_import("typing_extensions"),
])

print("\nOPTIONAL packages (affects which Parts run):")
has_hf   = try_import("transformers") and try_import("datasets")
has_med  = try_import("medmnist")
has_clip = try_import("open_clip_torch", "open_clip_torch")

print("\nParts availability:")
print(f"  Part A (Vision ViT)   : {'✓ available' if req else '✗ MISSING core deps'}")
print(f"  Part B (NLP BERT)     : {'✓ available' if has_hf else '✗ missing transformers/datasets (install Cell 1)'}")
print(f"  Part C (CLIP zero-shot): {'✓ available' if (has_med and has_clip) else '✗ missing medmnist or open_clip_torch (install Cell 1)'}")

if not req:
    raise RuntimeError("Missing required packages. Re-run Cell 1, then restart kernel and run all.")

print("\n✓ All required imports OK.")

REQUIRED packages:
  ✓ torch                          2.4.1+cu124
  ✓ torchvision                    0.19.1+cu124
  ✓ numpy                          2.4.4
  ✓ pandas                         3.0.2
  ✓ matplotlib                     3.10.8
  ✓ seaborn                        0.13.2
  ✓ scikit-learn                   1.8.0
  ✓ tqdm                           4.67.3
  ✓ typing_extensions              unknown

OPTIONAL packages (affects which Parts run):
  ✓ transformers                   5.5.3
  ✓ datasets                       4.8.4
  ✓ medmnist                       3.0.2
  ✗ open_clip_torch                NOT FOUND — re-run Cell 1

Parts availability:
  Part A (Vision ViT)   : ✓ available
  Part B (NLP BERT)     : ✓ available
  Part C (CLIP zero-shot): ✗ missing medmnist or open_clip_torch (install Cell 1)

✓ All required imports OK.


## Cell 6 — Matplotlib inline display configuration

**What this does:**
- Forces matplotlib to use the `Agg` backend (no display server needed on RunPod)
- ALL plots are saved as PNG files to the results directory
- After the experiments run, this cell provides a helper to display saved plots inline
- Sets matplotlib style for publication-quality figures

**Why `Agg` is needed on RunPod:**
RunPod containers do not have an X11 display server. Using the default `TkAgg` or  
`Qt5Agg` backend will crash with `cannot connect to X server`. `Agg` is headless and  
writes directly to PNG/PDF without needing a display.

In [6]:
import os
import matplotlib
matplotlib.use('Agg')  # MUST be set before any plt import — RunPod has no display
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Image as IPyImage

# Publication-quality style
plt.rcParams.update({
    'figure.dpi':       100,
    'savefig.dpi':      150,
    'font.size':        11,
    'axes.titlesize':   11,
    'axes.labelsize':   10,
    'legend.fontsize':  9,
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

RESULTS_DIR = Path(os.environ.get("ROBUST_NN_RESULTS_DIR", "/workspace/results_12April2026"))

def show_all_plots(pattern="*.png", max_plots=30):
    """Display all saved PNG plots inline after experiments complete."""
    plots = sorted(RESULTS_DIR.glob(pattern))
    if not plots:
        print(f"No plots found in {RESULTS_DIR}. Run the experiments first.")
        return
    print(f"Found {len(plots)} plots in {RESULTS_DIR}:")
    for i, p in enumerate(plots[:max_plots]):
        print(f"\n--- {p.name} ---")
        display(IPyImage(filename=str(p), width=900))
    if len(plots) > max_plots:
        print(f"\n... and {len(plots) - max_plots} more. Set max_plots higher if needed.")

def show_plot(filename):
    """Display a specific plot by filename."""
    p = RESULTS_DIR / filename
    if p.exists():
        display(IPyImage(filename=str(p), width=900))
    else:
        print(f"File not found: {p}")

print("✓ Matplotlib configured (Agg backend — headless, no X11 needed)")
print(f"  Plots will be saved to: {RESULTS_DIR}")
print("  Use show_all_plots() after experiments to view all figures inline.")

✓ Matplotlib configured (Agg backend — headless, no X11 needed)
  Plots will be saved to: /workspace/results_12April2026
  Use show_all_plots() after experiments to view all figures inline.


## Cell 7 — PART A: Vision ViT experiments (MNIST)

**What this runs:**
- Trains a Vision Transformer (ViT) from scratch on MNIST
- Tests all 9 loss functions: CCE, MAE, GCE, TruncGCE, SCE, DPD, SDIV, TSCCE, ForwardT
- Battery A+B: clean training + uniform label noise (η = 0, 0.1, 0.2, 0.3, 0.4)
- Battery C: FGSM adversarial attack (ε = 0, 1/255, 2/255, 4/255, 8/255)
- Battery E: SDIV (β, λ) 3D accuracy surface
- Curriculum GCE annealing experiment (novel contribution)

**Expected outputs:**
- `mnist_A_training_curves_s42.png` — per-loss training curves with correct Y-axis scales
- `mnist_confmat_*.png` — normalized confusion matrices (row = recall)
- `mnist_robustness_noise.png` — accuracy vs noise rate for all 9 losses
- `mnist_robustness_fgsm.png` — accuracy vs FGSM epsilon for all 9 losses
- `mnist_dual_frontier.png` — dual robustness scatter (novel figure)
- `mnist_sdiv_surface_3d.png` — 3D (β, λ) accuracy surface
- `mnist_curriculum_gce_eta0.3.png` — curriculum annealing plot

**Estimated time (RunPod L4 24GB GPU, QUICK_RUN=1, 30 epochs):** ~30–60 min  
**Estimated time (full paper run, 250 epochs):** ~6–10 hours  

**Progress bars** will appear for each loss × noise rate combination.

In [7]:
import runpy, os
import torch
from pathlib import Path

# Run only Part A for this cell
os.environ["ROBUST_NN_PART"] = "A"

print("Starting Part A: Vision ViT Experiments")
print(f"  Script: {SCRIPT}")
print(f"  GPU memory before: {torch.cuda.memory_allocated()/1024**3:.2f} GB allocated" if torch.cuda.is_available() else "  Running on CPU")
print("="*70)

try:
    runpy.run_path(str(SCRIPT), run_name="__main__")
    print("\n✓ Part A completed successfully.")
except Exception as e:
    import traceback
    print(f"\n✗ Part A failed with error: {type(e).__name__}: {e}")
    traceback.print_exc()
    print("\n--- Troubleshooting guide ---")
    print("CUDA out of memory → reduce VIT_BATCH in Cell 3 (try 128 or 64)")
    print("NCCL error        → single GPU is fine, NCCL only needed for multi-GPU")
    print("datasets error    → HF dataset download failed; check internet on pod")
    print("amp/autocast error → re-run Cell 4 to apply AMP patch, then retry")
finally:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"  GPU memory after (cache cleared): {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Starting Part A: Vision ViT Experiments
  Script: /workspace/12April2026_RobustNN_Experiments.py
  GPU memory before: 0.00 GB allocated
[Config] quick=False  part=A  vit_epochs=10  results=/workspace/results_12April2026
[Device] cuda  |  AMP=ON
         NVIDIA A40 | 44.4 GB VRAM


/workspace/12April2026_RobustNN_Experiments.py:48: UserWarning: torch 2.4.1+cu124 < 2.6 detected. Upgrade: pip install 'torch>=2.6'
  _w.warn(f"torch {_tcheck.__version__} < 2.6 detected. Upgrade: pip install 'torch>=2.6'",



  Robust NN Experiments — 12 April 2026
  Quick=False  Part=A
  Results → /workspace/results_12April2026/


  PART A: Vision ViT — MNIST
  Loading MNIST from ylecun/mnist ...


    converting:   0%|          | 0/60000 [00:00<?, ?it/s]

    converting:   0%|          | 0/10000 [00:00<?, ?it/s]

...

mnist|MAE|η=0.1:   0%|          | 0/10 [00:00<?, ?ep/s]

  [MAE] seed=42 η=0.1  best_acc=0.9747  time=2339s


mnist|GCE(q=0.7)|η=0.1:   0%|          | 0/10 [00:00<?, ?ep/s]

  GPU memory after (cache cleared): 0.02 GB


KeyboardInterrupt: 

## Cell 8 — PART B: NLP BERT fine-tuning experiment

**What this runs:**
- Fine-tunes `distilbert-base-uncased` on **dair-ai/emotion** (6-class emotion classification)
- Fine-tunes `allenai/scibert_scivocab_uncased` on **PubMedQA** (3-class yes/no/maybe)
- Tests all 9 robust loss functions for each dataset
- Uses AMP, gradient clipping, and linear warmup scheduler

**Prerequisites:**
- Requires `transformers` and `datasets` — Cell 2 will report if these are missing
- First run will download model weights (~250 MB for DistilBERT, ~430 MB for SciBERT)
- Weights cache in `/workspace/hf_cache/` for reuse on restart

**Expected outputs:**
- `nlp_Emotion_results.csv` — accuracy per loss for Emotion
- `nlp_PubMedQA_results.csv` — accuracy per loss for PubMedQA

**Estimated time (QUICK_RUN=1, 3 epochs, 1500 train samples):** ~20–40 min  
**Estimated time (full paper, 15 epochs, full data):** ~3–6 hours per dataset

**If HF_TOKEN is needed:**  
Set it in RunPod UI → Pod → Environment Variables as `HF_TOKEN=hf_yourtoken`

In [8]:
import runpy, os
import torch

# Run only Part B
os.environ["ROBUST_NN_PART"] = "B"

print("Starting Part B: NLP BERT Fine-tuning")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB allocated")
print("="*70)

try:
    runpy.run_path(str(SCRIPT), run_name="__main__")
    print("\n✓ Part B completed successfully.")
except ImportError as e:
    print(f"\n✗ ImportError: {e}")
    print("  → transformers/datasets not installed. Re-run Cell 1.")
except Exception as e:
    import traceback
    print(f"\n✗ Part B failed: {type(e).__name__}: {e}")
    traceback.print_exc()
    print("\n--- Troubleshooting guide ---")
    print("CUDA OOM       → reduce NLP_BATCH in Cell 3 (try 16 or 8)")
    print("HF auth error  → set HF_TOKEN in RunPod env vars")
    print("Dataset err    → PubMedQA may need HF_TOKEN; Emotion is public")
    print("Tokenizer err  → ensure transformers>=4.40 (see Cell 2)")
finally:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Starting Part B: NLP BERT Fine-tuning
  GPU memory: 0.02 GB allocated
[Config] quick=False  part=B  vit_epochs=10  results=/workspace/results_12April2026
[Device] cuda  |  AMP=ON
         NVIDIA A40 | 44.4 GB VRAM

  Robust NN Experiments — 12 April 2026
  Quick=False  Part=B
  Results → /workspace/results_12April2026/


  PART B: NLP BERT — Emotion
  Data: train=1500 val=2000 test=2000


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Emotion|CCE:   0%|          | 0/5 [00:00<?, ?it/s]

  Emotion|CCE: best_test_acc=0.8190


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Emotion|TSCCE:   0%|          | 0/5 [00:00<?, ?it/s]

  Emotion|TSCCE: best_test_acc=0.6975

  PART B: NLP BERT — PubMedQA
  Data: train=722 val=128 test=150

✗ Part B failed: ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

--- Troubleshooting guide ---
CUDA OOM       → reduce NLP_BATCH in Cell 3 (try 16 or 8)
HF auth error  → set HF_TOKEN in RunPod env vars
Dataset err    → PubMedQA may need HF_TOKEN; Emotion is public
Tokenizer err  → ensure transformers>=4.40 (see Cell 2)


Traceback (most recent call last):
  File "/tmp/ipykernel_1525/3856833162.py", line 14, in <module>
    runpy.run_path(str(SCRIPT), run_name="__main__")
  File "<frozen runpy>", line 291, in run_path
  File "<frozen runpy>", line 98, in _run_module_code
  File "<frozen runpy>", line 88, in _run_code
  File "/workspace/12April2026_RobustNN_Experiments.py", line 1436, in <module>
    run_nlp_battery(ds)
  File "/workspace/12April2026_RobustNN_Experiments.py", line 1185, in run_nlp_battery
    model = AutoModelForSequenceClassification.from_pretrained(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/models/auto/auto_factory.py", line 387, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py", line 4083, in from_pretrained
    config, dtype = _get_dtype(
                    ^^^^^^^^^^^
  Fil

## Cell 9 — PART C: CLIP zero-shot evaluation

**What this runs:**
- Zero-shot CLIP evaluation on **PathMNIST** and **DermaMNIST** histopathology datasets
- No training: encodes images and text prompts once, evaluates all 9 loss functions as metrics
- Uses `openai/clip-vit-base-patch32` from HuggingFace

**Prerequisites:**
- `medmnist` and `transformers` must be installed (Cell 1)
- CLIP model weights (~350 MB) downloaded once to `/workspace/hf_cache/`
- MedMNIST data (~4 MB per dataset) auto-downloaded

**Expected outputs:**
- `clip_PathMNIST_CLIP_results.csv` — loss values and accuracy per noise level
- `clip_DermaMNIST_CLIP_results.csv`
- `clip_PathMNIST_confmat.png` — normalized confusion matrix for zero-shot CLIP
- `clip_DermaMNIST_confmat.png`

**Estimated time:** ~5–15 min (no training, just inference)

In [9]:
import runpy, os
import torch

# Run only Part C
os.environ["ROBUST_NN_PART"] = "C"

print("Starting Part C: CLIP Zero-Shot Evaluation")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("="*70)

try:
    runpy.run_path(str(SCRIPT), run_name="__main__")
    print("\n✓ Part C completed successfully.")
except ImportError as e:
    print(f"\n✗ ImportError: {e}")
    print("  → medmnist or transformers missing. Re-run Cell 1.")
except Exception as e:
    import traceback
    print(f"\n✗ Part C failed: {type(e).__name__}: {e}")
    traceback.print_exc()
    print("\n--- Troubleshooting guide ---")
    print("medmnist error → pip install medmnist>=2.2.0")
    print("CLIP error     → pip install open_clip_torch>=2.24.0")
    print("PIL error      → pip install Pillow>=10.0.0")
finally:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Starting Part C: CLIP Zero-Shot Evaluation
[Config] quick=False  part=C  vit_epochs=10  results=/workspace/results_12April2026
[Device] cuda  |  AMP=ON
         NVIDIA A40 | 44.4 GB VRAM

  Robust NN Experiments — 12 April 2026
  Quick=False  Part=C
  Results → /workspace/results_12April2026/


  PART C: CLIP Zero-Shot — PathMNIST (CLIP)


 98%|█████████▊| 12435619840/12629854322 [19:15<00:05, 35137573.86it/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]


✗ Part C failed: ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

--- Troubleshooting guide ---
medmnist error → pip install medmnist>=2.2.0
CLIP error     → pip install open_clip_torch>=2.24.0
PIL error      → pip install Pillow>=10.0.0


Traceback (most recent call last):
  File "/tmp/ipykernel_1525/1049557499.py", line 13, in <module>
    runpy.run_path(str(SCRIPT), run_name="__main__")
  File "<frozen runpy>", line 291, in run_path
  File "<frozen runpy>", line 98, in _run_module_code
  File "<frozen runpy>", line 88, in _run_code
  File "/workspace/12April2026_RobustNN_Experiments.py", line 1441, in <module>
    run_clip_zero_shot_battery(ds, m)
  File "/workspace/12April2026_RobustNN_Experiments.py", line 1282, in run_clip_zero_shot_battery
    clip  = CLIPModel.from_pretrained(clip_id).to(DEVICE).eval()
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py", line 4083, in from_pretrained
    config, dtype = _get_dtype(
                    ^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py", line 786, in _get_dtype
    state_dict = load_state_dict(
                 ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/

## Cell 10 — Run ALL parts at once (A + B + C)

**Use this cell instead of Cells 7–9 if you want to run everything in one shot.**  
Equivalent to: `ROBUST_NN_PART=ABC python 12April2026_RobustNN_Experiments.py`

The script runs Parts A → B → C sequentially. GPU memory is cleared between parts.  
A progress summary is printed at the end.

**Recommended for:** Full overnight paper runs on RunPod A100/H100.

In [ ]:
import runpy, os, time
import torch

os.environ["ROBUST_NN_PART"] = "ABC"

print("Starting ALL PARTS (A + B + C)")
print(f"  Quick mode: {os.environ.get('ROBUST_NN_QUICK_RUN', '1')}")
print(f"  ViT epochs: {os.environ.get('ROBUST_NN_VIT_EPOCHS', '30')}")
print(f"  NLP epochs: {os.environ.get('ROBUST_NN_NLP_EPOCHS', '3')}")
print("="*70)

t0 = time.time()
try:
    runpy.run_path(str(SCRIPT), run_name="__main__")
    elapsed = time.time() - t0
    m, s = divmod(elapsed, 60)
    print(f"\n✓ ALL PARTS completed. Total time: {int(m)}m {s:.1f}s")
except Exception as e:
    import traceback
    elapsed = time.time() - t0
    m, s = divmod(elapsed, 60)
    print(f"\n✗ Failed after {int(m)}m {s:.1f}s")
    print(f"  Error: {type(e).__name__}: {e}")
    traceback.print_exc()
finally:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Starting ALL PARTS (A + B + C)
  Quick mode: 0
  ViT epochs: 10
  NLP epochs: 5
[Config] quick=False  part=ABC  vit_epochs=10  results=/workspace/results_12April2026
[Device] cuda  |  AMP=ON
         NVIDIA A40 | 44.4 GB VRAM

  Robust NN Experiments — 12 April 2026
  Quick=False  Part=ABC
  Results → /workspace/results_12April2026/


  PART A: Vision ViT — MNIST
  Loading MNIST from ylecun/mnist ...


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

    converting:   0%|          | 0/60000 [00:00<?, ?it/s]

    converting:   0%|          | 0/10000 [00:00<?, ?it/s]

  MNIST: train (60000, 1, 32, 32)  test (10000, 1, 32, 32)

  LOSS SCALE DIAGNOSTIC (C=10) — READ BEFORE PLOTTING
  Loss                   Scale / Y-axis range
  ------------------------------------------------------------------
  CCE                    [0, +∞), ~log(C) at init
  MAE                    [0, 1] always bounded
  GCE(q=0.7)             [0, 1.4285714285714286] bounded above by 1/q
  TruncGCE               [0, 1.4285714285714286] (subset of samples)
  SCE                    ≈α·log(C) + β·C at init — 10× amplified vs CCE!
  DPD                    (-∞, +∞), often negative — do NOT compare with CCE
  SDIV                   (-∞,+∞) A=0.240 B=0.810 — plot separately
  TSCCE                  [0, +∞), trimmed CCE — same units as CCE


  [Seed 42] Starting noise battery ...


mnist|CCE|η=0.0:   0%|          | 0/10 [00:00<?, ?ep/s]

## Cell 11 — List all output files

**What this does:**
- Lists every file in the results directory with size
- Categorizes by type: CSV data files vs PNG plots
- Run this after any experiment cell to check what was produced

**Expected file count after full A+B+C run:**
- ~25–40 PNG plots
- ~10–15 CSV files

In [ ]:
import os
from pathlib import Path

RESULTS_DIR = Path(os.environ.get("ROBUST_NN_RESULTS_DIR", "/workspace/results_12April2026"))

if not RESULTS_DIR.exists():
    print(f"Results directory does not exist yet: {RESULTS_DIR}")
    print("Run at least one experiment cell (7, 8, 9, or 10) first.")
else:
    all_files = sorted(RESULTS_DIR.iterdir())
    csvs  = [f for f in all_files if f.suffix == '.csv']
    plots = [f for f in all_files if f.suffix == '.png']
    other = [f for f in all_files if f.suffix not in ('.csv', '.png')]

    print(f"Results directory: {RESULTS_DIR}")
    print(f"Total files: {len(all_files)} ({len(csvs)} CSV, {len(plots)} PNG, {len(other)} other)")

    if csvs:
        print("\n--- CSV data files ---")
        for f in csvs:
            print(f"  {f.name:<55} {f.stat().st_size/1024:7.1f} KB")

    if plots:
        print("\n--- PNG plot files ---")
        for f in plots:
            print(f"  {f.name:<55} {f.stat().st_size/1024:7.1f} KB")

    if other:
        print("\n--- Other files ---")
        for f in other:
            print(f"  {f.name:<55} {f.stat().st_size/1024:7.1f} KB")

    total_mb = sum(f.stat().st_size for f in all_files) / 1024**2
    print(f"\nTotal size: {total_mb:.1f} MB")

## Cell 12 — Display all plots inline

**What this does:**
- Reads every saved PNG from the results directory
- Displays them inline in the notebook at full resolution
- Groups by experiment type for easy reading

**Key plots to look for:**
| Plot file | What to check |
|-----------|---------------|
| `mnist_A_training_curves_s42.png` | Each loss has its own Y-axis scale (the fix!) |
| `mnist_confmat_CCE_eta0_s42.png` | Values in [0,1] row-normalized (recall per class) |
| `mnist_robustness_noise.png` | Which loss degrades least as η increases |
| `mnist_robustness_fgsm.png` | Which loss is most adversarially robust |
| `mnist_dual_frontier.png` | Pareto frontier: top-right = best on BOTH metrics |
| `mnist_sdiv_surface_3d.png` | How accuracy varies over SDIV (β, λ) grid |
| `mnist_curriculum_gce_eta0.3.png` | Novel: q schedule + accuracy together |

In [ ]:
import os
from pathlib import Path
from IPython.display import display, Image as IPyImage

RESULTS_DIR = Path(os.environ.get("ROBUST_NN_RESULTS_DIR", "/workspace/results_12April2026"))

# Key plot groups to display
GROUPS = [
    ("Part A — Training curves (per-loss Y-axis)",   "*training_curves*"),
    ("Part A — Normalized confusion matrices",        "*confmat*"),
    ("Part A — Label noise robustness",               "*robustness_noise*"),
    ("Part A — Adversarial (FGSM) robustness",        "*robustness_fgsm*"),
    ("Part A — Dual robustness frontier (novel)",     "*dual_frontier*"),
    ("Part A — SDIV (β,λ) 3D surface",               "*sdiv_surface*"),
    ("Part A — Curriculum GCE annealing (novel)",     "*curriculum*"),
    ("Part C — CLIP zero-shot confusion matrices",    "clip_*confmat*"),
]

if not RESULTS_DIR.exists():
    print(f"No results yet. Run experiment cells first.")
else:
    displayed = set()
    for group_name, pattern in GROUPS:
        files = sorted(RESULTS_DIR.glob(pattern))
        if files:
            print(f"\n{'='*60}")
            print(f"  {group_name}")
            print(f"{'='*60}")
            for f in files:
                if f not in displayed:
                    print(f"  {f.name}")
                    display(IPyImage(filename=str(f), width=900))
                    displayed.add(f)

    # Any remaining plots not matched by groups
    remaining = [f for f in sorted(RESULTS_DIR.glob("*.png")) if f not in displayed]
    if remaining:
        print(f"\n{'='*60}")
        print("  Other plots")
        print(f"{'='*60}")
        for f in remaining:
            print(f"  {f.name}")
            display(IPyImage(filename=str(f), width=900))

## Cell 13 — Summary results table

**What this does:**
- Loads the noise robustness CSV and prints a structured summary table
- Shows accuracy for every (loss, noise_rate) combination
- Highlights the best performing loss at each noise level
- Also shows FGSM adversarial results

**How to read the table:**
- Each row is a (loss function, noise rate) pair
- Higher accuracy = better robustness at that corruption level
- SDIV should outperform CCE at high noise rates (η ≥ 0.2) — this is the paper's main claim

In [ ]:
import os
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path(os.environ.get("ROBUST_NN_RESULTS_DIR", "/workspace/results_12April2026"))

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.4f}".format)

for csv_name, title in [
    ("mnist_noise_results.csv", "MNIST — Label Noise Results"),
    ("mnist_fgsm_results.csv",  "MNIST — FGSM Adversarial Results"),
]:
    fpath = RESULTS_DIR / csv_name
    if fpath.exists():
        df = pd.read_csv(fpath)
        print(f"\n{'='*70}")
        print(f"  {title}")
        print(f"  {len(df)} rows  |  {fpath}")
        print(f"{'='*70}")
        # Average over seeds and pivot
        group_col = "noise_rate" if "noise_rate" in df.columns else "epsilon"
        agg = df.groupby(["loss", group_col])["accuracy"].mean().reset_index()
        pivot = agg.pivot(index="loss", columns=group_col, values="accuracy")
        print(pivot.to_string())
        # Best loss per column
        print(f"\nBest loss per {group_col}:")
        for col in pivot.columns:
            best_loss = pivot[col].idxmax()
            best_acc = pivot[col].max()
            print(f"  {group_col}={col:.4g}  →  {best_loss}  (acc={best_acc:.4f})")
    else:
        print(f"\nNot found: {fpath} — run Part A first (Cell 7 or 10).")

## Cell 14 — Generate LaTeX results table

**What this does:**
- Reads the noise robustness CSV and formats it as a LaTeX `booktabs` table
- Best result in each column is bolded automatically
- Ready to paste directly into the paper `.tex` file
- Saves as `results_table_noise.tex` in the results directory

**How to use:**
Copy the output and paste into your LaTeX paper under `\begin{table}[t]`.

In [ ]:
import os
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path(os.environ.get("ROBUST_NN_RESULTS_DIR", "/workspace/results_12April2026"))

def make_latex_table(csv_path: Path, caption: str, label: str,
                     group_col: str = "noise_rate") -> str:
    if not csv_path.exists():
        return f"% File not found: {csv_path}"

    df = pd.read_csv(csv_path)
    agg = df.groupby(["loss", group_col])["accuracy"].mean().reset_index()
    pivot = agg.pivot(index="loss", columns=group_col, values="accuracy")

    cols = list(pivot.columns)
    col_headers = [f"$\\eta={c:.2f}$" if group_col == "noise_rate"
                   else f"$\\varepsilon={c*255:.0f}/255$" for c in cols]

    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(f"\\caption{{{caption}}}")
    lines.append(f"\\label{{{label}}}")
    lines.append(r"\begin{tabular}{l" + "c" * len(cols) + "}")
    lines.append(r"\toprule")
    lines.append("Loss & " + " & ".join(col_headers) + r" \\\\ ")
    lines.append(r"\midrule")

    for loss_name, row in pivot.iterrows():
        cells = []
        for col_val, acc in zip(cols, row.values):
            if pd.isna(acc):
                cells.append("--")
            elif pivot[col_val].max() == acc:  # best in column → bold
                cells.append(f"\\textbf{{{acc:.4f}}}")
            else:
                cells.append(f"{acc:.4f}")
        lines.append(f"{loss_name} & " + " & ".join(cells) + r" \\\\ ")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


# Noise table
noise_tex = make_latex_table(
    RESULTS_DIR / "mnist_noise_results.csv",
    caption="Test accuracy under uniform label noise on MNIST. Best per column in \\textbf{bold}.",
    label="tab:noise_mnist",
    group_col="noise_rate",
)
print("--- Noise Robustness LaTeX Table ---")
print(noise_tex)
noise_tex_path = RESULTS_DIR / "results_table_noise.tex"
noise_tex_path.write_text(noise_tex)
print(f"\nSaved: {noise_tex_path}")

# FGSM table
fgsm_tex = make_latex_table(
    RESULTS_DIR / "mnist_fgsm_results.csv",
    caption="Test accuracy under FGSM adversarial attack on MNIST. Best per column in \\textbf{bold}.",
    label="tab:fgsm_mnist",
    group_col="epsilon",
)
print("\n--- FGSM Adversarial LaTeX Table ---")
print(fgsm_tex)
fgsm_tex_path = RESULTS_DIR / "results_table_fgsm.tex"
fgsm_tex_path.write_text(fgsm_tex)
print(f"\nSaved: {fgsm_tex_path}")

## Cell 15 — Troubleshooting: CUDA OOM recovery

**Run this cell ONLY if you got a CUDA Out-of-Memory (OOM) error.**

**What this does:**
- Clears all GPU memory caches
- Prints current memory usage
- Suggests reduced batch sizes for your GPU tier

**Common OOM causes and fixes:**
| Situation | Fix |
|-----------|-----|
| ViT batch=256, 16GB GPU | Set `ROBUST_NN_VIT_BATCH=128` in Cell 3 |
| BERT batch=32, 16GB GPU | Set `ROBUST_NN_NLP_BATCH=16` in Cell 3 |
| BERT batch=32, 8GB GPU | Set `ROBUST_NN_NLP_BATCH=8` and `NLP_MAX_LEN=64` |
| Running A+B+C together | Run Parts sequentially (Cell 7, 8, 9 separately) |

After running this cell, go back to Cell 3 and reduce the batch size, then rerun.

In [ ]:
import gc
import torch

# Force Python garbage collection
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    alloc = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    free = total - reserved

    print("GPU memory after clearing cache:")
    print(f"  Total    : {total:.2f} GB")
    print(f"  Allocated: {alloc:.2f} GB")
    print(f"  Reserved : {reserved:.2f} GB")
    print(f"  Free     : {free:.2f} GB")

    print("\nRecommended settings based on your GPU:")
    if total < 10:
        print("  8GB GPU: VIT_BATCH=64, NLP_BATCH=8, NLP_MAX_LEN=64")
    elif total < 20:
        print("  16GB GPU: VIT_BATCH=128, NLP_BATCH=16, NLP_MAX_LEN=128 (default)")
    elif total < 28:
        print("  24GB GPU: VIT_BATCH=256, NLP_BATCH=32, NLP_MAX_LEN=128 (paper settings)")
    else:
        print("  40GB+ GPU: VIT_BATCH=512, NLP_BATCH=64 (can increase for speed)")

    print("\nUpdate Cell 3 with these values, then rerun the experiment cell.")
else:
    print("No GPU. Running on CPU — reduce batch sizes for speed:")
    print("  os.environ['ROBUST_NN_VIT_BATCH'] = '32'")
    print("  os.environ['ROBUST_NN_NLP_BATCH'] = '8'")
    print("  os.environ['ROBUST_NN_VIT_EPOCHS'] = '3'  # for quick debug only")

## Cell 16 — FINAL: Archive results and prepare for download

**Run this cell BEFORE stopping the pod.**

**What this does:**
- Creates a timestamped ZIP archive of ALL results (CSV + PNG + TEX)
- Saves archive to `/workspace/` (outside the results folder, so it's easy to find)
- Prints the archive path and size
- **You MUST download this before stopping the RunPod pod** — storage is wiped on termination

**How to download from RunPod:**
1. In the RunPod dashboard, click your pod → **Files** tab
2. Navigate to `/workspace/`
3. Right-click `runpod_results_12April2026_*.zip` → Download

**Alternative:** Use `scp` or `rsync` from the RunPod terminal.

In [ ]:
import os
import shutil
import time
from pathlib import Path
from IPython.display import display, HTML

RESULTS_DIR = Path(os.environ.get("ROBUST_NN_RESULTS_DIR", "/workspace/results_12April2026"))

if not RESULTS_DIR.exists() or not any(RESULTS_DIR.iterdir()):
    print("Results directory is empty. Run experiments first (Cells 7–10).")
else:
    # Count files before archiving
    all_files = list(RESULTS_DIR.glob("**/*"))
    n_csv = sum(1 for f in all_files if f.suffix == '.csv')
    n_png = sum(1 for f in all_files if f.suffix == '.png')
    n_tex = sum(1 for f in all_files if f.suffix == '.tex')
    total_mb = sum(f.stat().st_size for f in all_files if f.is_file()) / 1024**2

    print(f"Archiving {len(all_files)} files ({n_csv} CSV, {n_png} PNG, {n_tex} TEX)")
    print(f"Total uncompressed: {total_mb:.1f} MB")

    # Create timestamped archive name
    ts = time.strftime("%Y%m%d_%H%M")
    archive_base = Path("/workspace") / f"runpod_results_12April2026_{ts}"

    archive_path = shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=RESULTS_DIR.parent,
        base_dir=RESULTS_DIR.name,
    )
    archive_size_mb = Path(archive_path).stat().st_size / 1024**2

    print(f"\n✓ Archive created: {archive_path}")
    print(f"  Size: {archive_size_mb:.1f} MB")
    print(f"\n{'='*60}")
    print("  IMPORTANT: Download this file before stopping the pod!")
    print(f"  RunPod Files tab → /workspace/ → {Path(archive_path).name}")
    print(f"{'='*60}")

    # Also display a clickable link if in JupyterLab
    display(HTML(
        f"<div style='background:#d4edda;padding:12px;border-radius:4px;'>"
        f"<b>Archive ready:</b> <code>{archive_path}</code><br>"
        f"Size: {archive_size_mb:.1f} MB | Files: {len(all_files)}<br>"
        f"<b>Download from RunPod Files browser before stopping the pod!</b>"
        f"</div>"
    ))